# 5.1 · 逻辑回归 + 评估指标 / Logistic Regression + Metrics

> **课程定位 / Where this fits**
> 第 1 课，**Part 5 · 监督学习：分类**。
> Lesson 1, **Part 5 · Supervised Classification**.
>
> Part 4 预测**连续值**（回归），从这一课起预测**离散类别**（分类）。逻辑回归是分类的"Hello World"，几乎所有分类思想都从它生长出来。
> Part 4 predicted **continuous values** (regression); from here we predict **discrete classes** (classification). Logistic regression is the "Hello World" of classification — nearly every classification idea grows from it.
>
> 这一课还顺带**建立整套分类评估指标**（混淆矩阵 / precision / recall / F1 / ROC / PR），因为没有指标就无法评估任何分类器，后面整个 Part 5 都要用。
> This lesson also builds the **full classification metrics toolkit** (confusion matrix / precision / recall / F1 / ROC / PR), since no classifier can be judged without them — all of Part 5 relies on these.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\mathbf{x}$ —— 一个样本的特征向量 / a sample's feature vector
> - $\mathbf{w}$ —— 权重向量（含偏置）/ weight vector (incl. bias)
> - $y\in\{0,1\}$ —— 真实类别 / true class
> - $p = \sigma(\mathbf{x}^\top\mathbf{w})$ —— 预测为正类的概率 / predicted probability of the positive class
> - $\sigma(z)=\dfrac{1}{1+e^{-z}}$ —— sigmoid 函数 / the sigmoid

> 💡 **面试相关 / Interview-relevant**
> - "逻辑回归为什么用 sigmoid + 交叉熵"（出镜率 ★★★★★，需手推 MLE）
> - "为什么不用 MSE 做分类"（★★★★★，非凸 + 梯度消失）
> - "precision / recall / F1 / ROC-AUC 何时用"（★★★★★）
> - "ROC-AUC vs PR-AUC"（★★★★，不平衡）
> - "逻辑回归是线性还是非线性模型"（★★★★，决策边界线性）
>
> Whiteboard hits: why sigmoid+cross-entropy (derive via MLE), why not MSE, when each metric applies, ROC vs PR, linear boundary.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 从**伯努利最大似然**推出逻辑回归（sigmoid + 交叉熵）。
   Derive logistic regression (sigmoid + cross-entropy) from **Bernoulli maximum likelihood**.
2. 解释为什么**不能用 MSE** 做分类。
   Explain why **MSE must not** be used for classification.
3. **从零**用梯度下降实现并与 sklearn 对照。
   Implement it **from scratch** with gradient descent and compare to sklearn.
4. 把**系数读成对数几率**，理解决策边界为何线性。
   Read **coefficients as log-odds** and see why the decision boundary is linear.
5. **建立完整分类指标体系**：混淆矩阵 / P / R / F1 / ROC-AUC / PR-AUC，并知道何时看哪个。
   Build the **full metrics toolkit** and know which to use when.

## 目录 / TOC
1. [先建直觉：从回归到分类](#1)
2. [从伯努利 MLE 到逻辑回归 ⭐](#2)
3. [为什么不用 MSE ⭐](#3)
4. [🩺 数据 + 从零实现](#4)
5. [决策边界 + 系数解读](#5)
6. [混淆矩阵 + P/R/F1 ⭐](#6)
7. [ROC 曲线与 AUC ⭐](#7)
8. [PR 曲线：不平衡的首选 ⭐](#8)
9. [阈值：概率→决策](#9)
10. [小结](#10)


<a id="1"></a>
## 1. 先建直觉：从回归到分类 / Intuition First

线性回归输出一个任意实数 $\mathbf{x}^\top\mathbf{w}$（可能是 −37，也可能是 +1000）。但分类要的是"属于正类的**概率**"，必须落在 0 和 1 之间。
Linear regression outputs any real number $\mathbf{x}^\top\mathbf{w}$ (could be −37 or +1000). But classification needs a **probability** of the positive class, which must lie between 0 and 1.

逻辑回归的全部巧思就是：**先算一个实数分数 $z=\mathbf{x}^\top\mathbf{w}$，再用 sigmoid 把它"压"进 (0,1)。** $z$ 很大 → 概率接近 1；$z$ 很小（很负）→ 概率接近 0；$z=0$ → 概率正好 0.5。
The whole trick of logistic regression: **compute a real score $z=\mathbf{x}^\top\mathbf{w}$, then squash it into (0,1) with the sigmoid.** Large $z$ → probability near 1; very negative $z$ → near 0; $z=0$ → exactly 0.5.

剩下两个问题，下面两节回答：(1) 用什么损失来训练？(2) 为什么不能像回归那样用平方误差？
Two questions remain, answered in the next two sections: (1) what loss do we train with? (2) why not square error like regression?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
z = np.linspace(-8, 8, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(z, 1/(1+np.exp(-z)), lw=2)
plt.axhline(0.5, color="gray", ls="--"); plt.axvline(0, color="gray", ls=":")
plt.xlabel("z = xᵀw (实数分数 real-valued score)")
plt.ylabel("σ(z) = 概率 probability")
plt.title("sigmoid 把任意实数压进 (0,1) / sigmoid squashes any real number into (0,1)")
plt.tight_layout(); plt.show()


<a id="2"></a>
## 2. 从伯努利 MLE 到逻辑回归 ⭐ / From Bernoulli MLE

损失函数不是拍脑袋定的，而是从**最大似然**自然推出来的（接 2.9 的 MLE）。
The loss isn't arbitrary — it falls out of **maximum likelihood** (continuing the MLE from 2.9).

每个样本的标签 $y\in\{0,1\}$ 是一次**伯努利试验**：正类概率为 $p$。所以观测到 $y$ 的概率可以写成一个紧凑式子：
Each label $y\in\{0,1\}$ is one **Bernoulli trial** with positive-class probability $p$. So the probability of observing $y$ can be written compactly:

$$\Pr(y\mid \mathbf{x}) = p^{\,y}(1-p)^{1-y}\qquad(y=1\text{ 时取 }p,\ y=0\text{ 时取 }1-p)$$

把所有样本的似然连乘、取负对数、求平均，就得到**二元交叉熵**损失：
Multiply the likelihood over all samples, take the negative log, and average — you get the **binary cross-entropy** loss:

$$J(\mathbf{w}) = -\frac{1}{n}\sum_i \big[\,y_i\log p_i + (1-y_i)\log(1-p_i)\,\big]$$

直觉：当真实是正类（$y=1$）时，损失是 $-\log p$——模型说概率越低，罚得越狠。
Intuition: when the truth is positive ($y=1$), the loss is $-\log p$ — the lower the model's claimed probability, the heavier the penalty.

**梯度异常简洁**（用到 0.8 算过的 sigmoid 导数性质）：
The **gradient is remarkably clean** (using the sigmoid-derivative property from 0.8):

$$\nabla_{\mathbf{w}} J = \frac{1}{n}\mathbf{X}^\top(\boldsymbol{p} - \mathbf{y})$$

它和线性回归的梯度 $\mathbf{X}^\top(\hat{\mathbf{y}}-\mathbf{y})$ **形式完全一样**——这不是巧合，而是广义线性模型(4.7)指数族的共同结构。
It has the **exact same form** as linear regression's gradient $\mathbf{X}^\top(\hat{\mathbf{y}}-\mathbf{y})$ — not a coincidence, but the shared GLM/exponential-family structure (4.7).


<a id="3"></a>
## 3. 为什么不用 MSE ⭐ / Why Not MSE

既然回归用平方误差(MSE)，分类为什么不能也用 $\frac12(\sigma(z)-y)^2$？两个致命问题（面试高频）：
Regression uses square error (MSE); why not use $\frac12(\sigma(z)-y)^2$ for classification too? Two fatal problems (frequently asked):

1. **非凸 / Non-convex**：MSE 套上 sigmoid 后，关于 $\mathbf{w}$ 是**非凸**的，梯度下降会卡在局部最优。交叉熵则是**凸函数**，保证全局最优。
   With sigmoid inside, MSE is **non-convex** in $\mathbf{w}$, so gradient descent gets stuck in local optima. Cross-entropy is **convex**, guaranteeing the global optimum.
2. **梯度消失 / Vanishing gradient**（0.11 实测过）：MSE+sigmoid 的梯度含因子 $\sigma'(z)=\sigma(1-\sigma)$。当模型**很自信却错了**（$z\to\pm\infty$）时 $\sigma'\to 0$，梯度消失，几乎学不动。交叉熵的梯度是干净的 $(p-y)$，**永不消失**。
   The MSE+sigmoid gradient contains $\sigma'(z)=\sigma(1-\sigma)$. When the model is **confidently wrong** ($z\to\pm\infty$), $\sigma'\to 0$, the gradient vanishes and learning stalls. Cross-entropy's gradient is the clean $(p-y)$, which **never vanishes**.

结论：**分类永远用交叉熵**。
Bottom line: **always use cross-entropy for classification.**


<a id="4"></a>
## 4. 数据 + 从零实现 / Data + From Scratch

我们用 **Breast Cancer Wisconsin**（威斯康星乳腺癌）数据集：569 个肿瘤样本，30 个由细胞图像算出的特征（半径、纹理、凹陷度等），标签是良性(1)/恶性(0)。是二分类的经典医学数据集。
We use the **Breast Cancer Wisconsin** dataset: 569 tumor samples, 30 features computed from cell images (radius, texture, concavity…), labeled benign (1) / malignant (0). A classic binary medical dataset.

逻辑回归（带正则）对尺度敏感，所以先标准化（3.4）。
Regularized logistic regression is scale-sensitive, so we standardize first (3.4).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)

data = load_breast_cancer()
X, y = data.data, data.target          # 0=恶性 malignant, 1=良性 benign
print(f"Breast Cancer: {X.shape}, 类别分布 class counts {np.bincount(y)} (良性 benign {y.mean():.0%})")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
sc = StandardScaler().fit(X_tr)        # 标准化 / standardize (3.4)
Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)


In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def fit_logreg(X, y, lr=0.1, n_iter=1000):
    Xb = np.c_[np.ones(len(X)), X]      # 加一列 1 当偏置 / add bias column
    w = np.zeros(Xb.shape[1])
    for _ in range(n_iter):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(y)   # 交叉熵梯度 cross-entropy gradient = Xᵀ(p-y)/n
        w -= lr * grad                   # 梯度下降一步 / one gradient-descent step
    return w

w = fit_logreg(Xtr, y_tr)
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(max_iter=2000).fit(Xtr, y_tr)   # 默认 L2; 数据近可分, 无正则系数会爆

# 系数量级取决于正则强度, 直接比"差异"无意义; 比方向(相关性)和预测一致性才公平
# coefficient magnitude depends on regularization; compare direction + predictions, not raw size
corr = np.corrcoef(w[1:], sk.coef_[0])[0, 1]
p_test = sigmoid(np.c_[np.ones(len(Xte)), Xte] @ w)
agree = ((p_test > 0.5).astype(int) == sk.predict(Xte)).mean()
print(f"从零 vs sklearn 系数方向相关性 coef-direction corr: {corr:.3f}  (≈1 表示同一方向)")
print(f"两者测试集预测一致率 prediction agreement:        {agree:.3f}")
print(f"从零模型 test 准确率 from-scratch test accuracy:   {((p_test > 0.5).astype(int) == y_te).mean():.3f}")


<a id="5"></a>
## 5. 决策边界 + 系数解读 / Decision Boundary & Coefficients

**决策边界是线性的**：预测正类当且仅当 $p>0.5$，即 $\mathbf{x}^\top\mathbf{w}>0$——这是一个**超平面**（二维里就是一条直线）。所以尽管 sigmoid 是非线性的，逻辑回归仍是**线性分类器**。
**The decision boundary is linear:** we predict positive iff $p>0.5$, i.e. $\mathbf{x}^\top\mathbf{w}>0$ — a **hyperplane** (a straight line in 2-D). So although the sigmoid is nonlinear, logistic regression is a **linear classifier**.


In [ ]:
# 2D 可视化决策边界 (取两个特征) / 2D decision boundary on two features
from sklearn.linear_model import LogisticRegression
X2 = Xtr[:, [0, 3]]      # mean radius, mean area (标准化后 standardized)
clf2 = LogisticRegression().fit(X2, y_tr)

xx, yy = np.meshgrid(np.linspace(X2[:,0].min(), X2[:,0].max(), 200),
                     np.linspace(X2[:,1].min(), X2[:,1].max(), 200))
Z = clf2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
cs = ax.contourf(xx, yy, Z, levels=20, cmap="RdBu", alpha=0.5)
ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=2)   # 决策边界=直线 boundary=line
ax.scatter(X2[:,0], X2[:,1], c=y_tr, cmap="RdBu", edgecolor="k", s=20)
plt.colorbar(cs, label="P(benign)"); ax.set_xlabel("mean radius"); ax.set_ylabel("mean area")
ax.set_title("决策边界=直线 / boundary is a straight line (P=0.5 contour)\nsigmoid 非线性但边界线性 → 线性分类器")
plt.tight_layout(); plt.show()


**系数 = 对数几率(log-odds)**。逻辑回归的系数有一个漂亮的解读：某特征增加 1 个单位，会让"正类的对数几率"增加这个系数；换成倍数就是 $e^{\text{系数}}$ 乘到**几率(odds)** 上。这是逻辑回归独有的可解释性。
**Coefficients = log-odds.** Logistic-regression coefficients have a neat reading: increasing a feature by one unit adds its coefficient to the **log-odds** of the positive class; equivalently it multiplies the **odds** by $e^{\text{coef}}$. This is logistic regression's signature interpretability.


In [ ]:
# 系数 = 对数几率 (log-odds) 解读 / coefficients as log-odds
sk = LogisticRegression(max_iter=2000).fit(Xtr, y_tr)
coef = pd.Series(sk.coef_[0], index=data.feature_names).sort_values(key=abs, ascending=False)
print("系数 Top 5 (标准化后) / top-5 coefficients (standardized):")
for name, c in coef.head(5).items():
    print(f"  {name:<22} {c:+.2f}  → 该特征 +1σ, 几率 odds ×{np.exp(c):.2f}")
print("\n💡 exp(系数) = 该特征+1单位时几率(odds)的倍数变化 / exp(coef) = odds multiplier per +1 unit")


<a id="6"></a>
## 6. 混淆矩阵 + P/R/F1 ⭐ / Confusion Matrix & P/R/F1

模型会预测，但**预测得好不好要用指标衡量**。一切从**混淆矩阵**开始——把"预测 vs 真实"的四种组合数出来：
A model predicts, but **how good those predictions are needs metrics**. It all starts with the **confusion matrix** — counting the four combinations of "predicted vs true":

| | 预测正 pred + | 预测负 pred − |
|---|---|---|
| **真实正 true +** | TP（真阳）| FN（假阴，漏报 miss）|
| **真实负 true −** | FP（假阳，误报 false alarm）| TN（真阴）|

由它派生出三个核心指标：
Three core metrics derive from it:

$$\text{Precision} = \frac{TP}{TP+FP}\ (\text{报警准不准 / how trustworthy are the alarms}),\quad
\text{Recall} = \frac{TP}{TP+FN}\ (\text{抓得全不全 / how many positives we caught})$$

$$F_1 = 2\cdot\frac{P\cdot R}{P+R}\ (\text{P 和 R 的调和平均 / harmonic mean of P and R})$$

**precision 和 recall 是一对权衡，由业务代价决定**：癌症筛查要高 recall（宁可误报也别漏诊），垃圾邮件过滤要高 precision（别把正常邮件误删）。
**Precision and recall trade off, decided by business cost:** cancer screening wants high recall (better a false alarm than a missed tumor); spam filtering wants high precision (don't delete real mail).


In [ ]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, accuracy_score)
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=2000).fit(Xtr, y_tr)
y_pred = clf.predict(Xte)

cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["恶性 malig","良性 benign"], yticklabels=["恶性 malig","良性 benign"], ax=ax)
ax.set_xlabel("预测 predicted"); ax.set_ylabel("真实 true"); ax.set_title("混淆矩阵 confusion matrix")
plt.tight_layout(); plt.show()

print(f"accuracy  = {accuracy_score(y_te, y_pred):.3f}")
print(f"precision = {precision_score(y_te, y_pred):.3f}")
print(f"recall    = {recall_score(y_te, y_pred):.3f}")
print(f"F1        = {f1_score(y_te, y_pred):.3f}")
print("\n💡 这里 1=良性; 把恶性预测成良性(FN)最危险, 实际应把'恶性'设为正类并优化 recall")
print("   Here 1=benign; predicting a malignant tumor as benign (FN) is the dangerous error.")


<a id="7"></a>
## 7. ROC 曲线与 AUC ⭐ / ROC Curve & AUC

precision/recall 都是在**某一个固定阈值(0.5)** 下算的。但模型其实输出的是连续概率，阈值可以变。**ROC 曲线扫遍所有阈值**，画出 **TPR(=recall) 对 FPR(=FP/(FP+TN))** 的轨迹：
Precision/recall are computed at **one fixed threshold (0.5)**. But the model outputs continuous probabilities, and the threshold can vary. The **ROC curve sweeps all thresholds**, plotting **TPR (=recall) against FPR (=FP/(FP+TN))**:

- 完美分类器直奔左上角（TPR=1, FPR=0）/ a perfect classifier hugs the top-left corner.
- 随机瞎猜落在对角线上 / random guessing lies on the diagonal.
- **AUC（曲线下面积）= 随机取一对(正,负)样本，模型给正样本更高分的概率**。0.5=瞎猜，1=完美。
  **AUC (area under the curve) = the probability the model ranks a random positive above a random negative.** 0.5 = chance, 1 = perfect.

AUC 的妙处：它衡量**排序能力**，与阈值无关、对类别比例不太敏感。
The beauty of AUC: it measures **ranking ability**, independent of threshold and fairly robust to class ratio.


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

y_proba = clf.predict_proba(Xte)[:, 1]
fpr, tpr, thresh = roc_curve(y_te, y_proba)
auc = roc_auc_score(y_te, y_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, label=f"逻辑回归 logreg (AUC={auc:.3f})")
ax.plot([0,1],[0,1],"k--", label="随机猜 chance (AUC=0.5)")
ax.scatter([0],[1], c="g", s=100, marker="*", label="完美 perfect", zorder=5)
ax.set_xlabel("FPR (假阳率 false-positive rate)"); ax.set_ylabel("TPR (真阳率=recall)")
ax.legend(); ax.set_title(f"ROC 曲线 ROC curve — AUC={auc:.3f}")
plt.tight_layout(); plt.show()
print(f"AUC = {auc:.3f} = 随机取一对(良性,恶性), 模型给良性更高分的概率 / ranking probability")
print("AUC 衡量排序, 与阈值无关 / AUC measures ranking, threshold-independent")


<a id="8"></a>
## 8. PR 曲线：不平衡的首选 ⭐ / PR Curve

**ROC 有个盲区**：当数据**极度不平衡**（正类很少）时，ROC 会**过度乐观**。因为 FPR 的分母 TN 极大，即使产生大量假阳，FPR 也只微微上升，ROC 看着还很漂亮。
**ROC has a blind spot:** when data is **highly imbalanced** (few positives), ROC is **over-optimistic**. Its FPR denominator TN is huge, so even many false positives barely move FPR, and the ROC still looks great.

**PR 曲线**（precision 对 recall）在不平衡下更诚实——它**完全不看 TN**，只盯着正类。所以**不平衡分类首选 PR-AUC**（即 average precision，接 3.11/5.14）。
The **PR curve** (precision vs recall) is more honest under imbalance — it **ignores TN entirely** and focuses on the positive class. So for **imbalanced problems, prefer PR-AUC** (a.k.a. average precision; see 3.11/5.14).


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

# 人为制造极端不平衡, 对比 ROC vs PR / extreme imbalance to contrast ROC vs PR
rng2 = np.random.default_rng(0)
n_pos, n_neg = 30, 2000
s_pos = rng2.normal(1.0, 1, n_pos)       # 正类分数略高 positives score a bit higher
s_neg = rng2.normal(0.0, 1, n_neg)
y_imb = np.r_[np.ones(n_pos), np.zeros(n_neg)]
s_imb = np.r_[s_pos, s_neg]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fpr, tpr, _ = roc_curve(y_imb, s_imb)
axes[0].plot(fpr, tpr); axes[0].plot([0,1],[0,1],"k--")
axes[0].set_title(f"ROC (AUC={roc_auc_score(y_imb, s_imb):.2f}) — 看着不错 looks fine")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")

prec, rec, _ = precision_recall_curve(y_imb, s_imb)
ap = average_precision_score(y_imb, s_imb)
axes[1].plot(rec, prec)
axes[1].axhline(n_pos/(n_pos+n_neg), color="k", ls="--", label=f"基线 baseline={n_pos/(n_pos+n_neg):.1%}")
axes[1].set_title(f"PR (AP={ap:.2f}) — 暴露真实困难 reveals difficulty")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"1.5% 正类: ROC-AUC={roc_auc_score(y_imb, s_imb):.2f}(乐观) 但 PR-AUC={ap:.2f}(真实困难)")
print("不平衡看 PR-AUC; ROC 被海量 TN 掩盖了 precision 问题 / use PR-AUC under imbalance")


### 指标选择速查 / Metric cheat-sheet
| 场景 / Scenario | 看什么 / Use |
|---|---|
| 类别均衡 + 关心整体对错 / balanced, overall correctness | accuracy + F1 |
| 漏报代价高（癌症/欺诈漏抓）/ misses are costly | **recall** |
| 误报代价高（垃圾邮件误杀）/ false alarms costly | **precision** |
| 综合排序能力（均衡）/ ranking, balanced | **ROC-AUC** |
| 综合排序能力（不平衡）/ ranking, imbalanced | **PR-AUC** ⭐ |
| 概率值要准（风控阈值）/ calibrated probabilities | 校准 calibration (5.15) |


<a id="9"></a>
## 9. 阈值：概率→决策 / Threshold: Probability → Decision

逻辑回归输出**概率**，默认用 0.5 当阈值变成 0/1 决策。但 0.5 是任意的——应**按业务成本调阈值**（接 2.10 期望损失的思想）。
Logistic regression outputs a **probability**, turned into a 0/1 decision at the default threshold 0.5. But 0.5 is arbitrary — **tune the threshold by business cost** (the expected-loss idea from 2.10).


In [ ]:
# 调阈值看 precision/recall 的权衡 / sweep threshold to see the P/R trade-off
thresholds = np.linspace(0.05, 0.95, 19)
precs = [precision_score(y_te, y_proba >= t, zero_division=1) for t in thresholds]
recs = [recall_score(y_te, y_proba >= t) for t in thresholds]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, precs, "o-", label="precision")
ax.plot(thresholds, recs, "s-", label="recall")
ax.axvline(0.5, color="gray", ls="--", label="默认 default 0.5")
ax.set_xlabel("阈值 threshold"); ax.legend()
ax.set_title("阈值权衡: 阈值↑ → precision↑ recall↓ / raising threshold trades recall for precision")
plt.tight_layout(); plt.show()
print("阈值低→抓全(高recall)但误报多; 阈值高→报准(高precision)但漏报多")
print("癌症筛查应降阈值(宁可误报也别漏诊) — 阈值是部署旋钮, 由代价决定 / threshold is set by cost")


<a id="10"></a>
## 10. 小结 / Summary

```
逻辑回归: p=σ(xᵀw); 损失=交叉熵(=伯努利 MLE, 2.9); 梯度=Xᵀ(p-y)/n
不用 MSE: 非凸 + 梯度消失 → 交叉熵凸 + 梯度(p-y)不消失(0.11)
决策边界线性 → 线性分类器(虽 sigmoid 非线性); 系数=对数几率, exp=odds 倍数
评估指标:
  混淆矩阵 → precision(报准) / recall(抓全) / F1(调和均值)
  ROC-AUC: 排序能力, 阈值无关; 均衡数据用
  PR-AUC : 不平衡首选(不看 TN)
  阈值: 概率→决策, 按业务成本调(2.10)
```

### 💡 面试速查 / Interview cheat-sheet
1. **逻辑回归 = sigmoid + 交叉熵**，由伯努利 MLE 推出。
   Logistic regression = sigmoid + cross-entropy, derived from Bernoulli MLE.
2. **不用 MSE**：非凸 + 梯度消失；交叉熵凸 + 梯度 $(p-y)$ 不消失。
   Not MSE: non-convex + vanishing gradient; cross-entropy is convex with non-vanishing $(p-y)$.
3. **决策边界线性** → 线性分类器；系数 = 对数几率。
   Linear boundary → linear classifier; coefficients = log-odds.
4. **precision vs recall** = 误报代价 vs 漏报代价的权衡。
   Precision vs recall = the false-alarm vs missed-detection cost trade-off.
5. **ROC-AUC 均衡用，PR-AUC 不平衡用**（ROC 被海量 TN 蒙蔽）。
   ROC-AUC when balanced, PR-AUC when imbalanced (ROC is fooled by huge TN).

### 下一节 / Next
**5.2 Softmax 回归**——逻辑回归处理二分类；多分类(≥3 类)用 softmax，把 sigmoid 推广成一个概率分布。
**5.2 Softmax Regression** — logistic handles 2 classes; for ≥3 classes, softmax generalizes the sigmoid into a probability distribution.
